A better version of `power_shendure_vs_minp.ipynb` with a more representative distribution of positive and negative effects. Specifically, we will be using the real values, plus many negatives.

Based on UKBB paper, expect ~30% of library to be active. So we will add 2x original size of negatives... 

We will also reduce computational burden by producing half-orthos.

# Imports & dask cluster creation

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

2026-01-23 14:32:44.104545: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-23 14:32:44.107563: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [3]:
client.dashboard_link

'http://127.0.0.1:8787/status'

# Ground truth creation

We will use parameter estimates from by_cell_type models as we expect these to be the most accurate.

In [4]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [5]:
primordial=scm.ortho.load(client,data_root/"shendure","ortho_primordial_v4")

In [6]:
primordial.compute_model_qc()

In [7]:
import pandas as pd

In [8]:
vals=[]
for key in primordial.by_cell_qc.keys():
    key="SurfaceEctoderm"
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
vals=pd.concat(vals)
vals=vals.groupby(["cre_id","cell_type"]).mean().reset_index()
vals

,cre_id,cell_type,mu
0,Bend5_chr4_8168,SurfaceEctoderm,0.019097
1,Bend5_chr4_8174,SurfaceEctoderm,0.01163
2,Bend5_chr4_8175,SurfaceEctoderm,0.671256
3,Bend5_chr4_8179,SurfaceEctoderm,0.019683
4,Bend5_chr4_8199,SurfaceEctoderm,0.004408
...,...,...,...
134,Txndc12_chr4_7978,SurfaceEctoderm,0.153156
135,eef1aP,SurfaceEctoderm,88.727592
136,pgk1P,SurfaceEctoderm,16.479346
137,reference,SurfaceEctoderm,0.007241


In [24]:
vals.dtypes

cre_id                     object
cell_type                  object
mu           Sparse[float64, 1.0]
dtype: object

In [25]:
vals["mu"] = vals["mu"].astype(float)

In [27]:
vals.dtypes

cre_id        object
cell_type     object
mu           float64
dtype: object

Now that we have reasonable mu estimates for the real CRE, let us add 200% "indistinguishable from minP".

In [9]:
#make "corresponding" inactive CREs...
mapping = {
    val: f"inactive_{i}"
    for i, val in enumerate(vals["cre_id"].unique())
}
mapping

{'Bend5_chr4_8168': 'inactive_0',
 'Bend5_chr4_8174': 'inactive_1',
 'Bend5_chr4_8175': 'inactive_2',
 'Bend5_chr4_8179': 'inactive_3',
 'Bend5_chr4_8199': 'inactive_4',
 'Btg1_chr10_9578': 'inactive_5',
 'Btg1_chr10_9588': 'inactive_6',
 'Btg1_chr10_9593': 'inactive_7',
 'Btg1_chr10_9612': 'inactive_8',
 'Btg1_chr10_9613': 'inactive_9',
 'Cdk5r1_chr11_12559': 'inactive_10',
 'Cdk5r1_chr11_12562': 'inactive_11',
 'Cdk5r1_chr11_12574': 'inactive_12',
 'Cdk5r1_chr11_12575': 'inactive_13',
 'Cdk5r1_chr11_12582': 'inactive_14',
 'Cdk5r1_chr11_12590': 'inactive_15',
 'Cited2_chr10_1253': 'inactive_16',
 'Cited2_chr10_1254': 'inactive_17',
 'Cited2_chr10_1267': 'inactive_18',
 'Col1a1_chr11_15258': 'inactive_19',
 'Col1a1_chr11_15259': 'inactive_20',
 'Col1a1_chr11_15270': 'inactive_21',
 'Col1a1_chr11_15275': 'inactive_22',
 'Col1a1_chr11_15276': 'inactive_23',
 'Col1a1_chr11_15301': 'inactive_24',
 'Col1a1_chr11_15307': 'inactive_25',
 'Col1a1_chr11_15316': 'inactive_26',
 'Col1a1_chr11_15

In [10]:
minP=scm.SHENDURE_BOUNDS.reference_activity
inactive=vals.copy().drop(columns=["mu"])
inactive["cre_id"] = inactive["cre_id"].map(mapping)
inactive["mu"]=minP
inactive

,cre_id,cell_type,mu
0,inactive_0,SurfaceEctoderm,0.019311
1,inactive_1,SurfaceEctoderm,0.019311
2,inactive_2,SurfaceEctoderm,0.019311
3,inactive_3,SurfaceEctoderm,0.019311
4,inactive_4,SurfaceEctoderm,0.019311
...,...,...,...
134,inactive_134,SurfaceEctoderm,0.019311
135,inactive_135,SurfaceEctoderm,0.019311
136,inactive_136,SurfaceEctoderm,0.019311
137,inactive_137,SurfaceEctoderm,0.019311


This is 100%. Let us double to 200%...

In [11]:
# duplicated version with _b appended
inactive_b = inactive.copy()
inactive_b["cre_id"] = inactive_b["cre_id"] + "_b"

# stack them
inactive_double = pd.concat([inactive, inactive_b], ignore_index=True)
inactive_double

,cre_id,cell_type,mu
0,inactive_0,SurfaceEctoderm,0.019311
1,inactive_1,SurfaceEctoderm,0.019311
2,inactive_2,SurfaceEctoderm,0.019311
3,inactive_3,SurfaceEctoderm,0.019311
4,inactive_4,SurfaceEctoderm,0.019311
...,...,...,...
273,inactive_134_b,SurfaceEctoderm,0.019311
274,inactive_135_b,SurfaceEctoderm,0.019311
275,inactive_136_b,SurfaceEctoderm,0.019311
276,inactive_137_b,SurfaceEctoderm,0.019311


Then stack with original gt...

In [12]:
final_gt=pd.concat([vals,inactive_double],ignore_index=True).rename({"mu":"true_mean"},axis=1)
final_gt

,cre_id,cell_type,true_mean
0,Bend5_chr4_8168,SurfaceEctoderm,0.019097
1,Bend5_chr4_8174,SurfaceEctoderm,0.01163
2,Bend5_chr4_8175,SurfaceEctoderm,0.671256
3,Bend5_chr4_8179,SurfaceEctoderm,0.019683
4,Bend5_chr4_8199,SurfaceEctoderm,0.004408
...,...,...,...
412,inactive_134_b,SurfaceEctoderm,0.019311
413,inactive_135_b,SurfaceEctoderm,0.019311
414,inactive_136_b,SurfaceEctoderm,0.019311
415,inactive_137_b,SurfaceEctoderm,0.019311


In [13]:
assert len(final_gt[["cre_id","cell_type"]].drop_duplicates()) == len(final_gt)

# Creating artificial libraries

In [14]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [15]:
libraries[2]

,cre_id,mpra_bc,abundance
0,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAA,0.000013
1,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAC,0.000002
2,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAG,0.000013
3,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAT,0.000047
4,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAACA,0.000024
...,...,...,...
55875,inactive_138_b,AAAAAAAAAAAATCGGCAAT,0.000010
55876,inactive_138_b,AAAAAAAAAAAATCGGCACA,0.000005
55877,inactive_138_b,AAAAAAAAAAAATCGGCACC,0.000009
55878,inactive_138_b,AAAAAAAAAAAATCGGCACG,0.000014


In [16]:
final_gt.dtypes

cre_id                     object
cell_type                  object
true_mean    Sparse[float64, 1.0]
dtype: object

# Creating sim

In [28]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-22",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=scm.SHENDURE_BOUNDS,
                            ground_truth=final_gt)

scMPRAforge: INFO: No 'state.parquet' found for 'twothird_pow_sim_2026-01-22'. Initalizing new object.


In [ ]:
sim.gamut()

scMPRAforge: INFO: A: drawn_library cols: Index(['Unnamed: 0', 'cre_id', 'mpra_bc', 'abundance'], dtype='object'), types: Unnamed: 0      int64
cre_id         object
mpra_bc        object
abundance     float64
dtype: object
scMPRAforge: INFO: A: cells_df cols: Index(['cell_type', 'cell_bc'], dtype='object'), types: cell_type    object
cell_bc      object
dtype: object
scMPRAforge: INFO: B: cells_df cols: Index(['cell_type', 'cell_bc', 'Unnamed: 0', 'cre_id', 'mpra_bc', 'abundance'], dtype='object'), types: cell_type      object
cell_bc        object
Unnamed: 0      int64
cre_id         object
mpra_bc        object
abundance     float64
dtype: object
scMPRAforge: INFO: A: drawn_library cols: Index(['Unnamed: 0', 'cre_id', 'mpra_bc', 'abundance'], dtype='object'), types: Unnamed: 0      int64
cre_id         object
mpra_bc        object
abundance     float64
dtype: object
scMPRAforge: INFO: A: cells_df cols: Index(['cell_type', 'cell_bc'], dtype='object'), types: cell_type    object
cell_

In [19]:
#sim.save()

In [20]:
#sim

Make the hypotheses...

In [21]:
#spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")
#hs_all_cre = scm.make_all_by_cre_hypotheses(
#    counts=demo_counts,
#    reference_cell_type="reference",
#)

In [22]:
sim.ground_truth.dtypes

cre_id             string[python]
cell_type          string[python]
true_mean    Sparse[float64, 1.0]
dtype: object

In [30]:
#client.close()
#cluster.close()